In [ ]:
MILVUS_HOST="10.18.27.59"
MILVUS_PORT="19530"
client = MilvusClient(
        host= MILVUS_HOST,
        port=  MILVUS_PORT,
        db_name = "tuan_milvus"
    )
res = client.list_collections()
print(res)

In [ ]:
from pymilvus import MilvusClient, Collection,connections
from langchain_huggingface import HuggingFaceEmbeddings
embedder = HuggingFaceEmbeddings(model_name="/home/huylnm/bge_m3_finetuned")

In [ ]:
query_vector=embedder.embed_query("cấp nước cam lâm")

In [ ]:
res = client.search(
    collection_name="hello67b6ea52972cb663a18a2d15",
    data=[query_vector],
    limit=50,
    # # highlight-start
    # filter='text like "red%" and likes > 50',
    output_fields=["text","document_id"]
    # highlight-end
)

for hits in res:
    print("TopK results:")
    for hit in hits:
        print(hit)

In [ ]:
def searchBacktracking( 
        results : list ,
        query_vector : list[float] ,
        collection_name : str ,
        limit : int = 3,
        offset : int = 1,
        lst_file : list = [],
        level_search : int = 2):
    # results = []
    # Điều kiện dừng đệ quy
    if level_search <= 0:
        return []

    # Chuẩn bị filter
    expr = f"file_name in {lst_file}" if lst_file else None
    filter_params = {"list_file": lst_file} if lst_file else None

    res = client.search(
        collection_name= collection_name,
        data=[query_vector],
        limit=limit,
        filter=expr,
        filter_params=filter_params,
        output_fields=["text","file_name","vector"],
        search_params={
            "metric_type": "L2",
            "offset" : offset, # bỏ qua offset phần tử đầu
        }
        
    )
    # print(res)


    new_results = []
    for hits in res:
        for hit in hits:
            if hit not in results:
                # print(hit["entity"]["file_name"])
                new_results.append(hit)
                next_res = searchBacktracking( 
                    results=new_results,
                    query_vector=hit["entity"]["vector"] ,
                    collection_name=collection_name,
                    limit=2 ,
                    offset=1 ,
                    lst_file=[hit["entity"]["file_name"]],
                    level_search= level_search - 1)
                new_results.extend(next_res)
    return new_results